In [1]:
import os

# Only move up if the current directory is 'research'
if os.path.basename(os.getcwd()) == "research":
    os.chdir("../")

print("Current directory:", os.getcwd())


Current directory: /home/sheshpal/Documents/ml_llmops_project/Data_science[1]-end-to-end/Data-science-Production


In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen= True)
class ModelTrainingConfig:
    root_dir: Path
    train_data_path:Path
    test_data_path:Path
    model_name:str
    alpha:float
    l1_ratio:float
    target_column:str


In [3]:
from src.Datascience.constants import CONFIG_FILE_PATH ,PARAMS_FILE_PATH,SCHEMA_FILE_PATH
from src.Datascience.utils.common import read_yaml, create_directories

In [7]:

class ConfigurationManager:
    def __init__(self,
            config_file_path = CONFIG_FILE_PATH,
            params_file_path = PARAMS_FILE_PATH,
            schema_file_path = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self)->ModelTrainingConfig:
        config = self.config.model_training
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN
        
        create_directories([config.root_dir])

        model_trainer_config = ModelTrainingConfig(
            root_dir=config.root_dir,
            train_data_path=config.train_data_path,
            test_data_path=config.test_data_path,
            model_name= config.model_name,
            alpha=params.alpha,
            l1_ratio=params.l1_ratio,
            target_column=schema.name

        )
        return model_trainer_config

In [8]:
from src.Datascience import logger
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.linear_model import ElasticNet
import joblib


In [13]:
class ModelTrainer:
    def __init__(self,config:ModelTrainingConfig):
        self.config = config

    def train(self):
        train_data= pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)

        train_x = train_data.drop([self.config.target_column], axis=1)
        test_x = test_data.drop([self.config.target_column], axis=1)
        train_y = train_data[self.config.target_column]
        test_y = test_data[self.config.target_column]


        lr = ElasticNet(alpha=self.config.alpha,l1_ratio=self.config.l1_ratio,random_state=42)
        lr.fit(train_x,train_y)

        joblib.dump(lr, os.path.join(self.config.root_dir, self.config.model_name))
        

In [14]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train()
except Exception as e:
    raise e

[2026-10-20 16:53:48,383:INFO:common:31:yaml file: config/config.yaml loaded successfully]
[2026-10-20 16:53:48,388:INFO:common:31:yaml file: params.yaml loaded successfully]
[2026-10-20 16:53:48,392:INFO:common:31:yaml file: schema.yaml loaded successfully]
[2026-10-20 16:53:48,394:INFO:common:59:Directory created: artifacts]
[2026-10-20 16:53:48,395:INFO:common:59:Directory created: artifacts/model_training]
